# PDE Discovery: Cahn-Hilliard Equation

f_t = -eps^2*Delta^2*f - Delta*f + 3*f^2*Delta*f + 6*f*|grad f|^2  (eps=0.08)

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
import derivatives
from library import build_equivariant_library, build_standard_library
from regression import TrainSTRidge
from utils import print_pde, rotation_90
np.random.seed(42); print('imports ok')

imports ok


In [2]:
eps=0.08; N=64; L=2*np.pi
x=np.linspace(0,L,N,endpoint=False)
X,Y=np.meshgrid(x,x,indexing='ij')
k=np.fft.fftfreq(N)*N*(2*np.pi/L)
KX,KY=np.meshgrid(k,k,indexing='ij')
K2=KX**2+KY**2; K4=K2**2
dt=0.005; T_end=30.; n_steps=int(T_end/dt); save_every=30
f0=0.02*np.random.randn(N,N)
denom=1.+dt*eps**2*K4
print('setup ok')

setup ok


In [3]:
snapshots=[]; times=[]
f=f0.copy()
for step in range(n_steps):
    fhat=np.fft.fft2(f)
    NL_hat=-K2*np.fft.fft2(f**3-f)
    fhat=(fhat+dt*NL_hat)/denom
    f=np.real(np.fft.ifft2(fhat))
    if step%save_every==0:
        snapshots.append(f.copy()); times.append(step*dt)
snapshots=np.array(snapshots); times=np.array(times)
print(f'Snapshots:{len(snapshots)}, f in [{snapshots.min():.3f},{snapshots.max():.3f}]')

Snapshots:200, f in [-1.042,1.025]


In [4]:
fig,axes=plt.subplots(1,4,figsize=(16,4))
for ax,idx in zip(axes,[0,len(snapshots)//4,len(snapshots)//2,-1]):
    im=ax.imshow(snapshots[idx],cmap='RdBu_r',vmin=-1,vmax=1,origin='lower',extent=[0,L,0,L])
    ax.set_title(f't={times[idx]:.1f}',fontsize=13); ax.axis('off')
fig.colorbar(im,ax=axes[-1],shrink=0.8)
fig.suptitle('Cahn-Hilliard phase separation',fontsize=14)
plt.tight_layout(); plt.savefig('../figures/cahn_hilliard_dynamics.pdf',bbox_inches='tight'); plt.close()
print('saved dynamics')

FileNotFoundError: [Errno 2] No such file or directory: '../figures/cahn_hilliard_dynamics.pdf'

In [ ]:
dt_snap=times[1]-times[0]
ft_snaps=(snapshots[2:]-snapshots[:-2])/(2*dt_snap)
f_snaps=snapshots[1:-1]; print(f'{len(f_snaps)} snapshots for regression')

In [ ]:
n_per_snap=200
Teq_rows_spec=[]; Tstd_rows_spec=[]; Ft_rows=[]
Teq_rows_poly=[]; Tstd_rows_poly=[]
np.random.seed(0)
for s in range(len(f_snaps)):
    idx=np.random.choice(N*N,n_per_snap,replace=False)
    Ft_rows.append(ft_snaps[s].ravel()[idx])
    with derivatives.use_backend('spectral'):
        Te,desc_eq=build_equivariant_library(f_snaps[s],L,L)
        Ts,desc_std=build_standard_library(f_snaps[s],L,L,max_order=4,max_poly=3)
    Teq_rows_spec.append(Te[idx]); Tstd_rows_spec.append(Ts[idx])
    with derivatives.use_backend('polynomial'):
        Te,_=build_equivariant_library(f_snaps[s],L,L)
        Ts,_=build_standard_library(f_snaps[s],L,L,max_order=4,max_poly=3)
    Teq_rows_poly.append(Te[idx]); Tstd_rows_poly.append(Ts[idx])
Ft_all=np.concatenate(Ft_rows)
Theta_eq_spec=np.vstack(Teq_rows_spec).astype(float)
Theta_std_spec=np.vstack(Tstd_rows_spec).astype(float)
Theta_eq_poly=np.vstack(Teq_rows_poly).astype(float)
Theta_std_poly=np.vstack(Tstd_rows_poly).astype(float)
print(f'Shape:{Theta_eq_spec.shape}')
print(f'Cond eq  : spectral={np.linalg.cond(Theta_eq_spec):.2e}  poly={np.linalg.cond(Theta_eq_poly):.2e}')
print(f'Cond std : spectral={np.linalg.cond(Theta_std_spec):.2e}  poly={np.linalg.cond(Theta_std_poly):.2e}')

In [ ]:
lam=1e-5; d_tol=0.05
print('=== Ground truth PDE ===')
print('f_t = -eps^2*Δ²f - Δf + 3f²Δf + 6f|∇f|²  (eps=0.08)\n')
print('── Spectral ──')
w_eq_spec=TrainSTRidge(Theta_eq_spec,Ft_all,lam,d_tol); print('Equivariant:'); print_pde(w_eq_spec,desc_eq)
w_std_spec=TrainSTRidge(Theta_std_spec,Ft_all,lam,d_tol); print('Standard:');   print_pde(w_std_spec,desc_std)
print('\n── Polynomial ──')
w_eq_poly=TrainSTRidge(Theta_eq_poly,Ft_all,lam,d_tol); print('Equivariant:'); print_pde(w_eq_poly,desc_eq)
w_std_poly=TrainSTRidge(Theta_std_poly,Ft_all,lam,d_tol); print('Standard:');   print_pde(w_std_poly,desc_std)

In [ ]:
true_eq=np.zeros(len(desc_eq))
true_eq[4]=-1.; true_eq[6]=3.; true_eq[8]=6.; true_eq[10]=-eps**2
act=np.abs(true_eq)>1e-10
print('--- Recovery summary (clean) ---')
for tag,w in [('Spectral  eq',w_eq_spec),('Polynomial eq',w_eq_poly)]:
    fp=int(np.sum((np.abs(w)>1e-10)&~act))
    fn=int(np.sum((np.abs(w)<1e-10)&act))
    err=float(np.max(np.abs(w[act]-true_eq[act])))
    print(f'  {tag}: fp={fp}, fn={fn}, max_coeff_err={err:.3e}')

In [ ]:
nl=0.02; snaps_n=snapshots+nl*snapshots.std()*np.random.randn(*snapshots.shape)
ft_n=(snaps_n[2:]-snaps_n[:-2])/(2*dt_snap); f_n=snaps_n[1:-1]
np.random.seed(0); Ftn=[]; Teqn_spec=[]; Tstdn_spec=[]; Teqn_poly=[]; Tstdn_poly=[]
for s in range(len(f_n)):
    idx=np.random.choice(N*N,n_per_snap,replace=False); Ftn.append(ft_n[s].ravel()[idx])
    with derivatives.use_backend('spectral'):
        Te,_=build_equivariant_library(f_n[s],L,L); Ts,_=build_standard_library(f_n[s],L,L,max_order=4,max_poly=3)
    Teqn_spec.append(Te[idx]); Tstdn_spec.append(Ts[idx])
    with derivatives.use_backend('polynomial'):
        Te,_=build_equivariant_library(f_n[s],L,L); Ts,_=build_standard_library(f_n[s],L,L,max_order=4,max_poly=3)
    Teqn_poly.append(Te[idx]); Tstdn_poly.append(Ts[idx])
Ftn=np.concatenate(Ftn)
Teqn_spec=np.vstack(Teqn_spec).astype(float); Tstdn_spec=np.vstack(Tstdn_spec).astype(float)
Teqn_poly=np.vstack(Teqn_poly).astype(float); Tstdn_poly=np.vstack(Tstdn_poly).astype(float)
print('── Spectral (noisy) ──')
w_eq_n_spec=TrainSTRidge(Teqn_spec,Ftn,lam,d_tol); w_std_n_spec=TrainSTRidge(Tstdn_spec,Ftn,lam,d_tol)
print('Equivariant:'); print_pde(w_eq_n_spec,desc_eq)
print('Standard:');   print_pde(w_std_n_spec,desc_std)
print('\n── Polynomial (noisy) ──')
w_eq_n_poly=TrainSTRidge(Teqn_poly,Ftn,lam,d_tol); w_std_n_poly=TrainSTRidge(Tstdn_poly,Ftn,lam,d_tol)
print('Equivariant:'); print_pde(w_eq_n_poly,desc_eq)
print('Standard:');   print_pde(w_std_n_poly,desc_std)
print('\n--- Noisy summary ---')
for tag,w in [('Spectral  eq',w_eq_n_spec),('Polynomial eq',w_eq_n_poly)]:
    fp=int(np.sum((np.abs(w)>1e-10)&~act)); fn=int(np.sum((np.abs(w)<1e-10)&act))
    err=float(np.max(np.abs(w[act]-true_eq[act])))
    print(f'  {tag}: fp={fp}, fn={fn}, max_coeff_err={err:.3e}')

In [ ]:
ftr=(rotation_90(snapshots[2:].transpose(1,2,0)).transpose(2,0,1)
     -rotation_90(snapshots[:-2].transpose(1,2,0)).transpose(2,0,1))/(2*dt_snap)
fr=rotation_90(snapshots[1:-1].transpose(1,2,0)).transpose(2,0,1)
np.random.seed(0); Ftr=[]; Teqr_spec=[]; Tstdr_spec=[]; Teqr_poly=[]; Tstdr_poly=[]
for s in range(len(fr)):
    idx=np.random.choice(N*N,n_per_snap,replace=False); Ftr.append(ftr[s].ravel()[idx])
    with derivatives.use_backend('spectral'):
        Te,_=build_equivariant_library(fr[s],L,L); Ts,_=build_standard_library(fr[s],L,L,max_order=4,max_poly=3)
    Teqr_spec.append(Te[idx]); Tstdr_spec.append(Ts[idx])
    with derivatives.use_backend('polynomial'):
        Te,_=build_equivariant_library(fr[s],L,L); Ts,_=build_standard_library(fr[s],L,L,max_order=4,max_poly=3)
    Teqr_poly.append(Te[idx]); Tstdr_poly.append(Ts[idx])
Ftr=np.concatenate(Ftr)
Teqr_spec=np.vstack(Teqr_spec).astype(float); Tstdr_spec=np.vstack(Tstdr_spec).astype(float)
Teqr_poly=np.vstack(Teqr_poly).astype(float); Tstdr_poly=np.vstack(Tstdr_poly).astype(float)
print('── Spectral (rotated) ──')
w_eq_r_spec=TrainSTRidge(Teqr_spec,Ftr,lam,d_tol); w_std_r_spec=TrainSTRidge(Tstdr_spec,Ftr,lam,d_tol)
print('Equivariant:'); print_pde(w_eq_r_spec,desc_eq)
print('Standard:');   print_pde(w_std_r_spec,desc_std)
print('\n── Polynomial (rotated) ──')
w_eq_r_poly=TrainSTRidge(Teqr_poly,Ftr,lam,d_tol); w_std_r_poly=TrainSTRidge(Tstdr_poly,Ftr,lam,d_tol)
print('Equivariant:'); print_pde(w_eq_r_poly,desc_eq)
print('Standard:');   print_pde(w_std_r_poly,desc_std)
print('\nCoefficient change under 90° rotation (clean):')
for tag,w,wr in [('Spectral  eq',w_eq_spec,w_eq_r_spec),('Polynomial eq',w_eq_poly,w_eq_r_poly),
                  ('Spectral  std',w_std_spec,w_std_r_spec),('Polynomial std',w_std_poly,w_std_r_poly)]:
    print(f'  {tag}: max|Δc|={np.max(np.abs(w-wr)):.2e}')

In [ ]:
bc=['tab:blue' if i in [4,6,8,10] and abs(c)>1e-10 else 'tab:red' if abs(c)>1e-10 else 'tab:gray'
    for i,c in enumerate(w_eq_spec)]
fig,axes=plt.subplots(1,2,figsize=(16,4))
ax=axes[0]; ax.bar(range(len(w_eq_spec)),np.real(w_eq_spec),color=bc)
ax.set_xticks(range(len(desc_eq))); ax.set_xticklabels(desc_eq,rotation=45,ha='right',fontsize=9)
ax.set_title('Equivariant coefficients'); ax.set_ylabel('value'); ax.axhline(0,color='k',lw=0.5)
ax=axes[1]; nz=np.abs(w_std_spec)>1e-10
if nz.any():
    ax.bar(np.where(nz)[0],np.real(w_std[nz]),color='tab:red')
    ax.set_xticks(np.where(nz)[0]); ax.set_xticklabels([desc_std[i] for i in np.where(nz)[0]],rotation=45,ha='right',fontsize=9)
ax.set_title('Standard coefficients (active only)'); ax.set_ylabel('value'); ax.axhline(0,color='k',lw=0.5)
plt.tight_layout(); plt.savefig('../figures/cahn_hilliard_coefficients.pdf',bbox_inches='tight'); plt.close()
print('saved coefficients')

In [ ]:
np.save('results/cahn_hilliard_results.npy',{
    'w_eq_spec':w_eq_spec,'w_std_spec':w_std_spec,
    'w_eq_poly':w_eq_poly,'w_std_poly':w_std_poly,
    'w_eq_noisy_spec':w_eq_n_spec,'w_std_noisy_spec':w_std_n_spec,
    'w_eq_noisy_poly':w_eq_n_poly,'w_std_noisy_poly':w_std_n_poly,
    'w_eq_rotated_spec':w_eq_r_spec,'w_std_rotated_spec':w_std_r_spec,
    'w_eq_rotated_poly':w_eq_r_poly,'w_std_rotated_poly':w_std_r_poly,
    'true_eq':true_eq,'desc_eq':desc_eq,'desc_std':desc_std,
},allow_pickle=True); print('Saved cahn_hilliard_results.npy')